# Notebook 1 of 3 — Training (manual §5.4 step 3: "Run training in one notebook")

**Settings:** Accelerator **GPU T4 x2**, Internet **On**. Start it with **Save Version → Save & Run All (Commit)**.

| Section | Manual task |
|---|---|
| Setup: clone, install, GPUs, versions, adapter tests | Task 1 (§7), §4.2 |
| Data preparation (SFT 30k/1k, HH 15k/1k, UltraFeedback 15k/1k) | §6.2–6.3 |
| SFT on Qwen2.5-0.5B | Task 3 (§9) |
| From-scratch DPO on HH-RLHF | Task 4 (§10) |
| From-scratch DPO on UltraFeedback | Task 4 (§10) |
| TRL DPOTrainer + LoRA persona run | Task 5 (§11) |

All checkpoints, `metrics.jsonl` files, plots and zips are written under
`/kaggle/working/cs5326-PA2-27100015/outputs/` (manual §5.5). When the run completes, this saved version is
attached as the input of Notebook 2.

## Task 1 — Repository and Kaggle setup

In [ ]:
# Manual section 7.1: clone the repository in Kaggle (Notebook settings: Internet ON).
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, show its output in this cell, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode} (see output above): {cmd}")

In [ ]:
# Manual sections 5.2 and 7.2: install dependencies without reinstalling PyTorch.
# peft/trl are added for the Task 5 LoRA run; torchao is removed to avoid the known PEFT conflict.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest "peft>=0.11.0" "trl>=0.24,<0.30"
!python -m pip install -q -e . --no-deps

In [ ]:
# Manual sections 5.1, 7.3 and 7.4: confirm two T4 GPUs and print package versions.
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
# Manual §4.2: public adapter smoke tests (plus our own edge-case and judging tests).
sh("PYTHONPATH=. python -m pytest -q tests/")

## Data preparation (§6.2–6.3, seed 42)

In [ ]:
# UltraChat single-turn SFT data, single-turn HH-RLHF pairs, single-turn UltraFeedback pairs.
# Writes data/processed/*.jsonl and data/manifests/*_ids.json + *_stats.json (filter counts for the report).
sh("python scripts/prepare_data.py --out_dir data/processed --which sft hh uf")
!wc -l data/processed/*.jsonl
!cat data/manifests/*_stats.json

In [ ]:
# One-minute end-to-end check with a tiny random model before the long runs.
sh("python scripts/smoke_train.py --work_dir /tmp/pa2_smoke")

## Task 3 — Supervised fine-tuning (`configs/sft.yaml`, guide configuration §9.2)

In [ ]:
sh("python scripts/train_sft.py --config configs/sft.yaml")

In [ ]:
!cat outputs/sft/summary.json
sh("python scripts/plot_metrics.py --run sft=outputs/sft --out_dir outputs/plots")
from IPython.display import Image, display
display(Image("outputs/plots/sft_loss.png"))

In [ ]:
!cd outputs && zip -q -r /kaggle/working/sft_run_outputs.zip sft -x "*.safetensors" "*.bin"
!ls -lh /kaggle/working/sft_run_outputs.zip && du -sh outputs/sft

## Task 4 — From-scratch DPO run 1: HH-RLHF (`configs/dpo_hh.yaml`, §10.3–10.4)

Policy = SFT `best` (fp32, `cuda:0`); frozen reference = SFT `best` (fp16, `cuda:1`); β = 0.1; LR 1e-6 fixed;
batch 1 × accumulation 32. Best checkpoint = highest validation reward accuracy.

In [ ]:
sh("python scripts/train_dpo.py --config configs/dpo_hh.yaml --set sft_checkpoint=outputs/sft/best")

In [ ]:
!cat outputs/dpo_hh/summary.json
sh("python scripts/plot_metrics.py --run dpo_hh=outputs/dpo_hh --out_dir outputs/plots")
from IPython.display import Image, display
display(Image("outputs/plots/dpo_hh_metrics.png"))

In [ ]:
!cd outputs && zip -q -r /kaggle/working/dpo_hh_run_outputs.zip dpo_hh -x "*.safetensors" "*.bin"
!ls -lh /kaggle/working/dpo_hh_run_outputs.zip && du -sh outputs/dpo_hh

## Task 4 — From-scratch DPO run 2: UltraFeedback (`configs/dpo_uf.yaml`, same code and settings)

In [ ]:
sh("python scripts/train_dpo.py --config configs/dpo_uf.yaml --set sft_checkpoint=outputs/sft/best")

In [ ]:
!cat outputs/dpo_uf/summary.json
sh("python scripts/plot_metrics.py --run dpo_uf=outputs/dpo_uf --out_dir outputs/plots")
from IPython.display import Image, display
display(Image("outputs/plots/dpo_uf_metrics.png"))

In [ ]:
!cd outputs && zip -q -r /kaggle/working/dpo_uf_run_outputs.zip dpo_uf -x "*.safetensors" "*.bin"
!ls -lh /kaggle/working/dpo_uf_run_outputs.zip && du -sh outputs/dpo_uf

## Task 5 — Idiot Sandwich: TRL DPOTrainer + LoRA (`configs/lora_dpo.yaml`)

`HuggingFaceTB/SmolLM2-360M-Instruct` on `data/persona_lora/ramsay_style_preferences.json` (22 rows held out).

In [ ]:
sh("python scripts/train_lora_dpo.py --config configs/lora_dpo.yaml")

In [ ]:
!cat outputs/lora_dpo/summary.json
sh("python scripts/plot_metrics.py --run lora_dpo=outputs/lora_dpo --out_dir outputs/plots")
from IPython.display import Image, display
display(Image("outputs/plots/lora_dpo_metrics.png"))

In [ ]:
!cd outputs && zip -q -r /kaggle/working/lora_dpo_run_outputs.zip lora_dpo -x "*.safetensors" "*.bin"
!ls -lh /kaggle/working/lora_dpo_run_outputs.zip && du -sh outputs/lora_dpo

## Training plots for the report (§14.4)

In [ ]:
sh("python scripts/plot_metrics.py --run sft=outputs/sft --run dpo_hh=outputs/dpo_hh "
   "--run dpo_uf=outputs/dpo_uf --run lora_dpo=outputs/lora_dpo --out_dir outputs/plots")
!cd outputs && zip -q -r /kaggle/working/plots.zip plots
from IPython.display import Image, display
display(Image("outputs/plots/dpo_runs_comparison.png"))

**Done.** This version's outputs are the input dataset for Notebook 2 (generation).